In [1]:
# Install all packages required by this notebook.
%pip install -q matplotlib tensorflow opencv-python scikit-learn

In [2]:
import kagglehub


# Download latest version
path = kagglehub.dataset_download("ajaysonicu/deepfakefusion-399k-realfake-faces")

print("Path to dataset files:", path)

100%|██████████| 6.67G/6.67G [06:09<00:00, 19.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ajaysonicu/deepfakefusion-399k-realfake-faces/versions/1


In [3]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization, GlobalAveragePooling2D

SEED = 42
# Seeds Python's `random`, NumPy and TensorFlow in one call
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
from pathlib import Path

dataset_root = Path(path) / "deepfakefusion_v2_sixsource_dataset"

print("Dataset root:", dataset_root)

for item in sorted(dataset_root.iterdir()):
    print("[DIR] " if item.is_dir() else "[FILE]", item.name)

for item in sorted(dataset_root.rglob("*")):
    relative_path = item.relative_to(dataset_root)

    if item.is_dir() and len(relative_path.parts) <= 3:
        print("[DIR] " if item.is_dir() else "[FILE]", relative_path)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_files = [
    file for file in dataset_root.rglob("*")
    if file.is_file() and file.suffix.lower() in image_extensions
]

print("Total images:", len(image_files))

for file in image_files[:20]:
    print(file.relative_to(dataset_root))

Dataset root: /root/.cache/kagglehub/datasets/ajaysonicu/deepfakefusion-399k-realfake-faces/versions/1/deepfakefusion_v2_sixsource_dataset
[DIR]  metadata
[DIR]  test
[DIR]  train
[DIR]  val
[DIR]  metadata
[DIR]  test
[DIR]  test/fake
[DIR]  test/real
[DIR]  train
[DIR]  train/fake
[DIR]  train/real
[DIR]  val
[DIR]  val/fake
[DIR]  val/real
Total images: 399992
test/real/test_real_000375953_new_140k_real_fake_faces.jpg
test/real/test_real_000228896_baseline_cdfv2.jpg
test/real/test_real_000034480_baseline_cdfv2.jpg
test/real/test_real_000236336_new_gravex_real.jpg
test/real/test_real_000075207_new_gravex_real.jpg
test/real/test_real_000240459_new_140k_real_fake_faces.jpg
test/real/test_real_000399215_baseline_dri.jpg
test/real/test_real_000295874_new_stylegan3_real.jpg
test/real/test_real_000325694_new_gravex_real.jpg
test/real/test_real_000348069_baseline_dri.jpg
test/real/test_real_000034324_baseline_dri.jpg
test/real/test_real_000165926_new_140k_real_fake_faces.jpg
test/real/test_

In [5]:
IMG_SIZE = (128, 128)
batch_size = 32

train_dir = dataset_root / "train"
val_dir = dataset_root / "val"
test_dir = dataset_root / "test"

# Class names come from the sub-folder names (alphabetical: fake=0, real=1),
# label_mode="binary" gives 0/1 float labels that match Dense(1, sigmoid).
train_full_ds = tf.keras.utils.image_dataset_from_directory(
    str(train_dir),
    image_size=IMG_SIZE,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

# Shuffle validation too: with shuffle=False, taking the first N batches
# would only return images from the first class folder ("fake").
val_full_ds = tf.keras.utils.image_dataset_from_directory(
    str(val_dir),
    image_size=IMG_SIZE,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

# Test set stays unshuffled so predictions line up with test_ds.file_paths
test_full_ds = tf.keras.utils.image_dataset_from_directory(
    str(test_dir),
    image_size=IMG_SIZE,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=False,
)

class_names = train_full_ds.class_names
print("Classes:", {name: i for i, name in enumerate(class_names)})
print("Training images:", len(train_full_ds.file_paths))
print("Validation images:", len(val_full_ds.file_paths))
print("Test images:", len(test_full_ds.file_paths))

Found 319994 files belonging to 2 classes.
Found 40000 files belonging to 2 classes.
Found 39998 files belonging to 2 classes.
Classes: {'fake': 0, 'real': 1}
Training images: 319994
Validation images: 40000
Test images: 39998


In [6]:
from IPython.display import clear_output

class LiveTrainingPlot(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.train_loss = []
        self.val_loss = []
        self.train_acc = []
        self.val_acc = []

        self.best_val_loss = float('inf')
        self.best_epoch = 0
        self.best_val_acc = 0

    def on_epoch_end(self, epoch, logs=None):

        # Current epoch metrics
        train_loss = logs['loss']
        val_loss = logs['val_loss']
        train_acc = logs['accuracy']
        val_acc = logs['val_accuracy']

        # Save epoch metrics
        self.train_loss.append(logs['loss'])
        self.val_loss.append(logs['val_loss'])
        self.train_acc.append(logs['accuracy'])
        self.val_acc.append(logs['val_accuracy'])

        # Check if this is the best epoch so far
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.best_val_acc = val_acc
            self.best_epoch = epoch + 1

        best_epoch_idx = self.best_epoch - 1

        # Clear previous plot
        clear_output(wait=True)
        epochs_completed = range(1, len(self.train_loss) + 1)
        plt.figure(figsize=(12, 5))

        # Loss curves
        plt.subplot(1, 2, 1)
        plt.plot(
            epochs_completed,
            self.train_loss,
            marker='o',
            label='Training Loss'
        )
        plt.plot(
            epochs_completed,
            self.val_loss,
            marker='o',
            label='Validation Loss'
        )
        plt.xlabel('Epoch')
        plt.ylabel('Binary Cross-Entropy Loss')
        plt.title('Training vs Validation Loss')
        plt.legend()
        plt.grid(True)
        # Accuracy curves
        plt.subplot(1, 2, 2)
        plt.plot(
            epochs_completed,
            self.train_acc,
            marker='o',
            label='Training Accuracy'
        )
        plt.plot(
            epochs_completed,
            self.val_acc,
            marker='o',
            label='Validation Accuracy'

        )
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.title('Training vs Validation Accuracy')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()



        print(f"Previous EPOCH: {epoch + 1}")
        print(f"Train Loss: {train_loss:.4f} | "f"Train Accuracy: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | "f"Val Accuracy:   {val_acc:.4f}")

        print(f"\nBEST EPOCH SO FAR: {self.best_epoch}")
        print(f"Train Loss: {self.train_loss[best_epoch_idx]:.4f} | "f"Train Accuracy: {self.train_acc[best_epoch_idx]:.4f}")
        print(f"Val Loss: {self.best_val_loss:.4f} | "f"Val Accuracy: {self.best_val_acc:.4f}")

    def on_train_end(self, logs=None):
        print(f"Training completed for {len(self.train_loss)} epochs")
        print(f"Best Epoch: {self.best_epoch}\n" f"Best Validation Loss: {self.best_val_loss:.4f}\n" f"Validation Accuracy at Best Epoch: {self.best_val_acc:.4f}")

        print("\nEarlyStopping restored the model weights " f"from Epoch {self.best_epoch}.")

In [ ]:
# Sampling targets for the training and validation pipelines
# resource: https://www.geeksforgeeks.org/machine-learning/python-image-classification-using-keras/

train_sample_size = 319994
validation_sample_size = 40000
epochs = 100 # probably gonna lower it to 50 or 60, but for now let's see how it goes

input_shape = (*IMG_SIZE, 3)   # image_dataset_from_directory always yields channels-last

AUTOTUNE = tf.data.AUTOTUNE

!pip install -q keras-tuner
import keras_tuner as kt
import math
import numpy as np

# Rescale pixels to [0, 1]
normalize = lambda x, y: (x / 255.0, y)

# Random flips/rotation/contrast, only applied to training data, to help the model generalize
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomContrast(0.1),
])

# .take() counts batches, not images. The file list was shuffled once with SEED,
# so the first N batches are a random sample. .cache() then freezes that sample
# so every epoch sees the same images (~2 GB of RAM for the training subset;
# lower train_sample_size if that is too much).
train_ds = (train_full_ds
            .take(math.ceil(train_sample_size / batch_size))
            .map(normalize, num_parallel_calls=AUTOTUNE)
            .cache()
            .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))

val_ds = (val_full_ds
          .take(math.ceil(validation_sample_size / batch_size))
          .map(normalize, num_parallel_calls=AUTOTUNE)
          .cache()
          .prefetch(AUTOTUNE))

# Full test set, same preprocessing, for model.evaluate() later
test_ds = test_full_ds.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Bayesian hyperparameter search. kernel size, first filter count, and learning rate
# stay fixed at values already justified by the manual tuning from before, only the later filters/dense size/dropout are searched
def build_model(hp):
    filters_2 = hp.Choice('filters_2', [64, 128])
    filters_3 = hp.Choice('filters_3', [128, 256])
    dense_units = hp.Choice('dense_units', [64, 128, 256])
    dropout_rate = hp.Float('dropout_rate', 0.3, 0.6, step=0.1)

    model = Sequential()
    model.add(Conv2D(32, (3, 3), padding="same", input_shape=input_shape))
    model.add(BatchNormalization())  # stabilizes training after each conv layer
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(filters_2, (3, 3), padding="same"))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(filters_3, (3, 3), padding="same"))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())
    model.add(Dense(dense_units))
    model.add(Activation('relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(1))
    model.add(Activation('sigmoid'))

    model.compile(loss='binary_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),  # Higher initial LR which anneals as val loss plateaus
                  metrics=['accuracy'])
    return model

tuner = kt.BayesianOptimization(
    build_model,
    objective='val_loss',
    max_trials=25,
    directory='tuner_results',
    project_name='deepfake_cnn_tuning'
)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
    ]
)

best_hp = tuner.get_best_hyperparameters(1)[0]
print("Best hyperparameters:", best_hp.values)

# k-fold validation of the best config only not every trial, saves some time
k_folds = 5

# Data aug only in training
cv_pool_ds = (train_full_ds
              .take(math.ceil(train_sample_size / batch_size))
              .concatenate(val_full_ds.take(math.ceil(validation_sample_size / batch_size)))
              .map(normalize, num_parallel_calls=AUTOTUNE)
              .unbatch())

fold_val_losses = []
fold_val_accuracies = []

for fold in range(k_folds):
    fold_val_ds = (cv_pool_ds.enumerate()
                   .filter(lambda i, data: i % k_folds == fold)
                   .map(lambda i, data: data)
                   .batch(batch_size)
                   .prefetch(AUTOTUNE))

    fold_train_ds = (cv_pool_ds.enumerate()
                      .filter(lambda i, data: i % k_folds != fold)
                      .map(lambda i, data: data)
                      .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
                      .batch(batch_size)
                      .prefetch(AUTOTUNE))

    fold_model = tuner.hypermodel.build(best_hp)
    fold_model.fit(fold_train_ds, validation_data=fold_val_ds, epochs=30,
                    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
                    verbose=0)

    val_loss, val_acc = fold_model.evaluate(fold_val_ds, verbose=0)
    fold_val_losses.append(val_loss)
    fold_val_accuracies.append(val_acc)
    print(f"Fold {fold+1}/{k_folds}: val_loss={val_loss:.4f}, val_accuracy={val_acc:.4f}")

print(f"\nK-Fold CV results for best hyperparameters:")
print(f"Mean val_loss: {np.mean(fold_val_losses):.4f} (+/- {np.std(fold_val_losses):.4f})")
print(f"Mean val_accuracy: {np.mean(fold_val_accuracies):.4f} (+/- {np.std(fold_val_accuracies):.4f})")

Trial 25 Complete [00h 02m 19s]
val_loss: 0.5158016681671143

Best val_loss So Far: 0.5013221502304077
Total elapsed time: 00h 50m 57s
Best hyperparameters: {'filters_2': 128, 'filters_3': 128, 'dense_units': 256, 'dropout_rate': 0.3}


ValueError: in user code:

    File "/tmp/ipykernel_5590/2884535129.py", line 126, in None  *
        lambda x, y: (data_augmentation(x, training=True), y)
    File "/usr/local/lib/python3.13/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler  **
        raise e.with_traceback(filtered_tb) from None
    File "/usr/local/lib/python3.13/dist-packages/keras/src/models/functional.py", line 278, in _adjust_input_rank
        raise ValueError(

    ValueError: Exception encountered when calling Sequential.call().
    
    [1mInvalid input shape for input Tensor("args_0:0", shape=(128, 128, 3), dtype=float32) with name 'keras_tensor' and path ''. Expected shape (None, 128, 128, 3), but input has incompatible shape (128, 128, 3)[0m
    
    Arguments received by Sequential.call():
      • inputs=tf.Tensor(shape=(128, 128, 3), dtype=float32)
      • training=True
      • mask=None
      • kwargs=<class 'inspect._empty'>


In [8]:
best_epoch = np.argmin(history.history['val_loss'])

print(f"Best Epoch: {best_epoch + 1}")
print(f"Best Training Loss: {history.history['loss'][best_epoch]}")
print(f"Best Validation Loss: {history.history['val_loss'][best_epoch]}")
print(f"Best Training Accuracy: {history.history['accuracy'][best_epoch]}")
print(f"Best Validation Accuracy: {history.history['val_accuracy'][best_epoch]}")


NameError: name 'history' is not defined